# 05 — Geometric ablations (Table 2 / Table 7)

Rerun pre-training with alternative curvature pairings to confirm `(H, S)` is optimal.

| Variant | trees factor | cycles factor |
|---------|--------------|---------------|
| H×S (default) | hyperbolic κ=-1 | spherical κ=+1 |
| H×H | hyperbolic | hyperbolic |
| S×S | spherical | spherical |
| R×S | Euclidean stand-in | spherical |
| H×R | hyperbolic | Euclidean stand-in |

We simulate 'Euclidean' by setting curvature to ±0.05 (near-flat) since the unified CCS formalism does not admit κ=0.

In [ ]:
import os, sys, yaml, copy
REPO = '/content/QuantAI'
sys.path.insert(0, os.path.join(REPO, 'src'))
os.environ['RIEMANN_GFM_DATA_ROOT'] = '/content/drive/MyDrive/RiemannGFM/data'
%cd $REPO
base_cfg = yaml.safe_load(open('configs/pretrain.yaml'))
variants = {
    'H_S': (-1.0, 1.0),
    'H_H': (-1.0, -1.0),
    'S_S': (1.0, 1.0),
    'Rnear_S': (-0.05, 1.0),
    'H_Rnear': (-1.0, 0.05),
}

In [ ]:
os.makedirs('configs/ablation', exist_ok=True)
for name, (kh, ks) in variants.items():
    c = copy.deepcopy(base_cfg)
    c['model']['kappa_H'] = kh
    c['model']['kappa_S'] = ks
    path = f'configs/ablation/{name}.yaml'
    yaml.safe_dump(c, open(path, 'w'))
    ckpt = f'/content/drive/MyDrive/RiemannGFM/checkpoints/ablation_{name}.pt'
    !python main.py pretrain --config $path --checkpoint $ckpt
    for ds in ['citeseer', 'pubmed', 'airports']:
        !python main.py lp --dataset $ds --checkpoint $ckpt